[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap09_series_temporais/previsao_defasagens_comparacao_temporal_autoveiculos.ipynb)

# Previsão com atributos defasados e comparação entre abordagens

Este notebook usa a série mensal da base SériesTemporais_Autoveículos para comparar um modelo baseado em atributos defasados, um modelo clássico ARIMA/SARIMA e uma referência ingênua. A comparação usa o mesmo horizonte e o mesmo período de teste. O treino e a validação respeitam a ordem temporal.

Caminho presumido no repositório: `capitulos/cap09_series_temporais/previsao_defasagens_comparacao_temporal_autoveiculos.ipynb`.

## 1. Objetivo e desenho experimental

Atributos defasados descrevem valores anteriores da própria série. Para prever o mês t, cada atributo deve conter apenas valores observados até t-1, sem usar informação futura. A seleção de atributos e hiperparâmetros ocorre por validação temporal com janelas expansivas no treinamento. O período final de teste é reservado antes dessas decisões e é usado uma única vez na comparação final.

As abordagens comparadas são: (1) regressão com atributos defasados e tendência/calendário opcional, (2) modelo clássico ARIMA ou SARIMA, e (3) referência ingênua que repete o último valor observado. Todas produzem previsões para exatamente o mesmo período de teste. MAE e RMSE são calculados na unidade original.

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEMENTE=2026
np.random.seed(SEMENTE)
URL_BASE="https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/S%C3%A9riesTemporais_Autove%C3%ADculos.xlsx"

## 2. Carregamento e configuração da série

Inspecione as abas e colunas, e configure os nomes exatos. Defina também uma regra de agregação mensal coerente com a medida. O notebook verifica continuidade mensal e não interpola lacunas automaticamente.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE,timeout=30) as resposta:
            return pd.ExcelFile(io.BytesIO(resposta.read()))
    except Exception as erro:
        print("Download indisponível:",erro)
        try:
            from google.colab import files
            enviados=files.upload()
            if not enviados: raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel=carregar_excel()
bases={aba:pd.read_excel(excel,sheet_name=aba) for aba in excel.sheet_names}
for aba,df in bases.items():
    print(f"\n{aba}: {df.shape}")
    print(list(df.columns))
    display(df.head())

In [ ]:
ABA=None
COLUNA_DATA=None
COLUNA_VALOR=None
COLUNA_CATEGORIA=None
FILTRO_CATEGORIA=None
AGREGACAO="sum"  # escolher 'sum' ou 'mean' e justificar

if not all([ABA,COLUNA_DATA,COLUNA_VALOR]) or ABA not in bases:
    raise ValueError("Configure ABA, COLUNA_DATA e COLUNA_VALOR após inspecionar a planilha.")
df=bases[ABA].copy()
if COLUNA_CATEGORIA and FILTRO_CATEGORIA:
    df=df[df[COLUNA_CATEGORIA].astype(str).str.contains(str(FILTRO_CATEGORIA),case=False,na=False,regex=False)].copy()
if COLUNA_DATA not in df.columns or COLUNA_VALOR not in df.columns: raise KeyError("Confira os nomes das colunas configuradas.")
df[COLUNA_DATA]=pd.to_datetime(df[COLUNA_DATA],errors="coerce",dayfirst=True)
df[COLUNA_VALOR]=pd.to_numeric(df[COLUNA_VALOR],errors="coerce")
df=df.dropna(subset=[COLUNA_DATA,COLUNA_VALOR]).copy()
if AGREGACAO not in ["sum","mean"]: raise ValueError("AGREGACAO deve ser 'sum' ou 'mean'.")
df["mes"]=df[COLUNA_DATA].dt.to_period("M")
print("Contagem de registros mensais:");display(df.groupby("mes").size().describe().to_frame("n"))
serie=df.groupby("mes")[COLUNA_VALOR].agg(AGREGACAO).sort_index()
serie=serie.reindex(pd.period_range(serie.index.min(),serie.index.max(),freq="M"))
serie.index=serie.index.to_timestamp(how="start")
if serie.isna().any(): raise ValueError("Há meses ausentes. Investigue antes da modelagem, sem interpolar automaticamente.")
if len(serie)<60: raise ValueError("Série curta para teste e validação temporal com horizonte anual; revise o período.")
print("Observações:",len(serie),"| período:",serie.index.min(),"a",serie.index.max())
serie.plot(figsize=(12,4),title="Série mensal selecionada");plt.show()

## 3. Teste final e validação temporal

O horizonte final de teste é definido antes das escolhas de atributos e modelos. O padrão é 12 meses. As janelas de validação são expansivas: cada origem usa somente observações anteriores à janela validada, e a previsão é feita para o horizonte completo. A validação usa as mesmas origens e horizonte para comparar as abordagens. O período de teste não participa da seleção.

In [ ]:
HORIZONTE=12
N_JANELAS=3
if len(serie)<=HORIZONTE+2*HORIZONTE: raise ValueError("Dados insuficientes para teste anual e múltiplas janelas.")
treino_teste=serie.iloc[:-HORIZONTE].copy()
teste=serie.iloc[-HORIZONTE:].copy()
# Origem comum: cada janela de validação tem o mesmo horizonte.
primeira_origem=len(treino_teste)-N_JANELAS*HORIZONTE
origens=[primeira_origem+i*HORIZONTE for i in range(N_JANELAS)]
if min(origens)<24: raise ValueError("Treino inicial de validação curto; reduza N_JANELAS com justificativa.")
print("Teste final:",teste.index.min().date(),"a",teste.index.max().date())
print("Origens de validação:",[treino_teste.index[o-1].date() for o in origens])

## 4. Construção dos atributos sem informação futura

Para cada mês t, `lag_1` corresponde ao valor de t-1; `lag_2`, ao valor de t-2, e assim sucessivamente. A média móvel defasada é calculada sobre valores anteriores e deslocada, para não incorporar o próprio valor que se pretende prever. O mês do calendário pode ser incluído como atributo conhecido previamente.

Para previsões de vários passos sem atualização no horizonte, atributos defasados posteriores ao primeiro passo precisam usar previsões anteriores, e não valores observados do teste. A função de previsão abaixo implementa essa atualização recursiva. Em cada origem, o modelo é treinado apenas com linhas cujos alvos já são conhecidos naquela origem.

In [ ]:
N_LAGS=12
USAR_CALENDARIO=True

# Atributos são determinísticos e usam apenas passado ou calendário conhecido.
def construir_matriz(y,n_lags=12,usar_calendario=True):
    dados=pd.DataFrame({"y":y})
    for lag in range(1,n_lags+1): dados[f"lag_{lag}"]=dados["y"].shift(lag)
    dados["media_3_lagged"]=dados["y"].shift(1).rolling(3).mean()
    dados["media_12_lagged"]=dados["y"].shift(1).rolling(12).mean()
    if usar_calendario:
        dados["mes_sin"]=np.sin(2*np.pi*dados.index.month/12)
        dados["mes_cos"]=np.cos(2*np.pi*dados.index.month/12)
    return dados.dropna()

def previsao_lags_recursiva(historico,horizonte,origem_modelo,n_lags=12,usar_calendario=True):
    treino_features=construir_matriz(historico,n_lags,usar_calendario)
    colunas=[c for c in treino_features.columns if c!="y"]
    modelo=make_pipeline(StandardScaler(),Ridge(alpha=1.0))
    modelo.fit(treino_features[colunas],treino_features["y"])
    valores=list(historico.astype(float).values)
    previsoes=[]
    datas=pd.date_range(historico.index[-1]+pd.offsets.MonthBegin(1),periods=horizonte,freq="MS")
    for data in datas:
        linha={f"lag_{lag}":valores[-lag] for lag in range(1,n_lags+1)}
        linha["media_3_lagged"]=float(np.mean(valores[-3:]))
        linha["media_12_lagged"]=float(np.mean(valores[-12:]))
        if usar_calendario:
            linha["mes_sin"]=np.sin(2*np.pi*data.month/12)
            linha["mes_cos"]=np.cos(2*np.pi*data.month/12)
        x=pd.DataFrame([linha],columns=colunas)
        yhat=float(modelo.predict(x)[0])
        previsoes.append(yhat)
        valores.append(yhat)
    return pd.Series(previsoes,index=datas)

print("Atributos:",list(construir_matriz(treino_teste).columns))

## 5. Modelos comparados e ajuste temporal

O modelo de atributos defasados é uma regressão Ridge com padronização dos preditores. A regularização reduz a instabilidade quando lags são correlacionados. O modelo clássico usa uma especificação ARIMA configurável; acrescente componentes sazonais se justificados na análise anterior, usando somente o treinamento da respectiva origem. A referência simples repete o último valor conhecido na origem.

As ordens do modelo clássico e os parâmetros dos atributos devem ser fixados antes da avaliação final. A validação temporal permite comparar as opções usando exclusivamente janelas anteriores ao teste.

In [ ]:
# Defina a especificação clássica a partir do exercício anterior e da análise do treinamento.
ARIMA_ORDER=(1,1,0)
SARIMA_SEASONAL_ORDER=(0,0,0,0)  # para sazonalidade justificada, use (P,D,Q,12)

def metricas(y,yp):
    return {"MAE":mean_absolute_error(y,yp),"RMSE":np.sqrt(mean_squared_error(y,yp))}

def previsao_classica(historico,horizonte):
    modelo=ARIMA(historico,order=ARIMA_ORDER,seasonal_order=SARIMA_SEASONAL_ORDER,
                 enforce_stationarity=False,enforce_invertibility=False).fit()
    pred=modelo.get_forecast(steps=horizonte).predicted_mean
    datas=pd.date_range(historico.index[-1]+pd.offsets.MonthBegin(1),periods=horizonte,freq="MS")
    return pd.Series(np.asarray(pred),index=datas)

def previsao_ingenua(historico,horizonte):
    datas=pd.date_range(historico.index[-1]+pd.offsets.MonthBegin(1),periods=horizonte,freq="MS")
    return pd.Series(float(historico.iloc[-1]),index=datas)

# Validação: previsões para cada janela, com horizonte fixo e origens comuns.
resultados_validacao=[]
for numero,origem in enumerate(origens,1):
    hist=treino_teste.iloc[:origem]
    alvo=treino_teste.iloc[origem:origem+HORIZONTE]
    if len(alvo)!=HORIZONTE: continue
    preds={
        "Defasagens + Ridge":previsao_lags_recursiva(hist,HORIZONTE,hist.index[-1],N_LAGS,USAR_CALENDARIO),
        "ARIMA/SARIMA":previsao_classica(hist,HORIZONTE),
        "Ingênua":previsao_ingenua(hist,HORIZONTE)
    }
    for nome,pred in preds.items():
        m=metricas(alvo,pred)
        resultados_validacao.append({"janela":numero,"modelo":nome,**m})
validacao=pd.DataFrame(resultados_validacao)
display(validacao.groupby("modelo")[["MAE","RMSE"]].mean().sort_values("MAE").round(3))
display(validacao.pivot(index="janela",columns="modelo",values="MAE").round(3))

## 6. Escolha prévia e avaliação final no mesmo teste

Escolha a abordagem de atributos defasados e fixe os parâmetros antes de abrir a avaliação final. Para manter a comparação justa, os três métodos usam o mesmo histórico de treinamento, as mesmas datas de teste e o mesmo horizonte. A escolha entre especificações deve ser baseada na validação temporal e no diagnóstico, não nos erros do teste.

In [ ]:
# A especificação de lags foi fixada acima. Ajuste o modelo final apenas no treino_teste.
# A referência ingênua também usa somente o último valor do treino_teste.
pred_lags=previsao_lags_recursiva(treino_teste,HORIZONTE,treino_teste.index[-1],N_LAGS,USAR_CALENDARIO)
pred_classica=previsao_classica(treino_teste,HORIZONTE)
pred_ingenua=previsao_ingenua(treino_teste,HORIZONTE)

# Alinhar datas explicitamente ao teste, evitando qualquer deslocamento temporal.
pred_lags.index=teste.index
pred_classica.index=teste.index
pred_ingenua.index=teste.index
comparacao=pd.DataFrame({"observado":teste,"Defasagens + Ridge":pred_lags,
                         "ARIMA/SARIMA":pred_classica,"Ingênua":pred_ingenua})
metricas_teste=pd.DataFrame({nome:metricas(teste,comparacao[nome])
                             for nome in ["Defasagens + Ridge","ARIMA/SARIMA","Ingênua"]}).T
metricas_teste.index.name="Abordagem"
display(metricas_teste.sort_values("MAE").round(3))
ax=treino_teste.plot(figsize=(12,5),label="Treinamento")
teste.plot(ax=ax,label="Observado: teste",color="black")
pred_lags.plot(ax=ax,label="Defasagens + Ridge",linestyle="--")
pred_classica.plot(ax=ax,label="ARIMA/SARIMA",linestyle="--")
pred_ingenua.plot(ax=ax,label="Ingênua",linestyle=":")
ax.set_title("Comparação no mesmo período de teste e horizonte");ax.legend();plt.show()
display(comparacao.round(2))

## 7. Interpretação e limites

A validação em janelas temporais estima o desempenho em origens anteriores ao teste, enquanto a comparação final fornece uma avaliação no período reservado. MAE e RMSE são sensíveis ao horizonte, à escala e ao intervalo escolhido. Diferenças pequenas entre métodos podem não ser estáveis em outros períodos.

O modelo com lags depende da escolha dos atributos, da regularização e da estratégia recursiva para previsões de múltiplos passos. A referência ingênua é simples, mas constitui um critério importante. O modelo clássico depende da especificação ARIMA/SARIMA e de seus pressupostos. O resultado não demonstra superioridade universal de uma abordagem.

### Checklist

- Unidade temporal, agregação e continuidade foram verificadas?
- Teste foi separado antes de qualquer seleção?
- Lags e médias móveis usam apenas valores anteriores ao alvo?
- Previsões multi-step não reutilizam valores observados do teste?
- Validação tem origens temporais ordenadas e mesmo horizonte entre modelos?
- ARIMA/SARIMA e referência ingênua usam os mesmos cortes?
- MAE e RMSE são calculados para as mesmas datas do teste?
- As limitações de um único período final e das escolhas de especificação foram registradas?